#### Imports and setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.config import RAW_DATA_FILE, TARGET_COLUMN
from src.data_loader import load_energy_dataset, parse_datetime_column
from src.feature_engineering_utils import (
    clip_outliers_iqr,
    select_columns,
    create_time_features,
    create_lag_features,
    create_rolling_features,
    create_interaction_features,
    drop_feature_engineering_nulls,
    save_dataframe,
)

pd.set_option("display.max_columns", None)

## Feature Engineering

This notebook creates additional predictive features for the appliance energy forecasting task.  
The engineered features include time-based features, lagged features, rolling statistics, and interaction features.

#### 1) Load Dataset

In [ ]:
raw_df = load_energy_dataset(RAW_DATA_FILE)
raw_df = parse_datetime_column(raw_df, datetime_column="date", set_as_index=True)

raw_df.head()

##### 2) Feature Selection

In [ ]:
# dataset shape 
print("Shape before feature engineering:", raw_df.shape)

In [ ]:
selected_cols = ["T1", 
                 "RH_1", 
                 "Appliances",
                 "lights",
                 "T_out",
                 "RH_out",
                 "Windspeed",
                 "Visibility",
                 "Press_mm_hg",
                 ]

energy_df = select_columns(raw_df, selected_cols)

##### 3) Outlier Treatment
- (missing value checking part skipped as it was previously done with the feature-engineered dataset)

In [ ]:
outlier_columns = [
    "Appliances",
    "lights",
    "T_out",
    "RH_out",
    "Windspeed",
]

energy_df = clip_outliers_iqr(
    energy_df=energy_df,
    columns=outlier_columns,
    iqr_multiplier=1.5,
)

energy_df.head()

Outlier Treatment
- Outliers were treated using IQR-based clipping for selected columns rather than row deletion.  
- This approach preserves the time-series continuity while reducing the influence of extreme values.

#### 4) Creating time-based features

In [ ]:
energy_df = create_time_features(energy_df)

energy_df[
    ["Appliances", "hour", "day_of_week", "month", "day_of_month", "is_weekend"]
].head()

- Time-related variables were created to capture temporal usage patterns in appliance energy consumption.  
- These include hour of the day, day of the week, month, day of the month, and a weekend indicator.

#### 5) Creating lag features

In [ ]:
lag_steps = [1, 3, 6] # 10mins, 30mins, 1hr

energy_df = create_lag_features(
    energy_df=energy_df,
    target_column=TARGET_COLUMN,
    lag_steps=lag_steps,
)

energy_df[
    ["Appliances", "Appliances_lag_1", "Appliances_lag_3", "Appliances_lag_6"]
].head(10)


- Lagged target features were created to capture temporal dependency in appliance energy consumption.  
- These features allow the model to use recent historical energy usage when predicting future consumption.

#### 6) Creating rolling features

In [ ]:
window_sizes = [6, 12] # 1h rolling mean, 2h rolling mean

energy_df = create_rolling_features(
    energy_df=energy_df,
    target_column=TARGET_COLUMN,
    window_sizes=window_sizes,
)

energy_df[
    [
        "Appliances",
        "Appliances_rolling_mean_6",
        "Appliances_rolling_mean_12",
    ]
].head(15)

- Rolling mean features were generated to smooth short-term fluctuations and capture local energy consumption trends.  
- These features help represent recent average behavior rather than relying only on single past points.

#### 7) Creating interaction features

In [ ]:
energy_df = create_interaction_features(energy_df)

interaction_columns = [
    column_name
    for column_name in energy_df.columns
    if "interaction" in column_name
]

energy_df[interaction_columns].head()

- Interaction terms were created to capture combined effects between temperature and humidity variables.  
- These combined environmental conditions may influence appliance energy use more effectively than the original variables alone.

#### 8) Checks

In [ ]:
# checking for missing values after feature eng process
energy_df.isnull().sum()[energy_df.isnull().sum() > 0]

In [ ]:
# removing rows with null values (caused by the lag or rolling)
energy_df = drop_feature_engineering_nulls(energy_df)

print("Shape after dropping nulls:", energy_df.shape)
energy_df.head()

#### 9) Saving feature-engineered dataset

In [ ]:
save_dataframe(
    energy_df=energy_df,
    file_name="feature_engineered_energy_data.csv",
)